# E-commerce Purchase Prediction
Predicting 7-day purchases from 42M October and 67M November events.

## 0. Setup
The heavy CSV processing runs only when RUN_DATA_PREP is True. Otherwise, the notebook reads the cleaned Parquet files from a saved Kaggle dataset.

In [38]:
!pip install -q duckdb

import os
import duckdb

# Set to True only if the cleaned Parquet files ever need to be rebuilt from the raw CSVs
RUN_DATA_PREP = False

con = duckdb.connect()
con.sql("SET memory_limit='20GB'")
con.sql("SET threads=4")
con.sql("SET temp_directory='/tmp/duckdb'")

def find_file(name):
    # Look in this session's output first, then in attached datasets
    for base in ['/kaggle/working', '/kaggle/input']:
        for root, _, files in os.walk(base):
            if name in files:
                return os.path.join(root, name)
    return None

# Quick check: should print the paths of the three files from your new dataset
print(find_file('oct.parquet'))
print(find_file('oct_clean.parquet'))
print(find_file('nov_clean.parquet'))

/kaggle/working/oct.parquet
/kaggle/working/oct_clean.parquet
/kaggle/working/nov_clean.parquet


In [39]:
# Check that the dataset copies exist in /kaggle/input
for name in ['oct.parquet', 'oct_clean.parquet', 'nov_clean.parquet']:
    found = [os.path.join(r, name) for r, _, f in os.walk('/kaggle/input') if name in f]
    print(name, found)

oct.parquet ['/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct.parquet', '/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct.parquet']
oct_clean.parquet ['/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct_clean.parquet', '/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct_clean.parquet']
nov_clean.parquet ['/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/nov_clean.parquet', '/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/nov_clean.parquet']


In [40]:
# Convert the raw October CSV to Parquet (runs only when rebuilding data)
if RUN_DATA_PREP:
    src = find_file('2019-Oct.csv')
    con.sql(f"""
        COPY (SELECT * FROM read_csv_auto('{src}'))
        TO '/kaggle/working/oct.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    print('October converted')

In [41]:
con.sql(f"CREATE OR REPLACE VIEW oct AS SELECT * FROM '{find_file('oct.parquet')}'")

In [42]:
# Remove exact duplicates and add derived columns (runs only when rebuilding data)
if RUN_DATA_PREP:
    con.sql("""
        COPY (
            SELECT DISTINCT
                event_time,
                CAST(event_time AS DATE)              AS event_date,
                HOUR(event_time)                      AS event_hour,
                DAYOFWEEK(event_time)                 AS day_of_week,
                event_type, product_id, category_id, category_code,
                split_part(category_code, '.', 1)     AS category_l1,
                COALESCE(brand, 'unknown')            AS brand,
                price, user_id, user_session
            FROM oct
        ) TO '/kaggle/working/oct_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

con.sql(f"CREATE OR REPLACE VIEW oct_clean AS SELECT * FROM '{find_file('oct_clean.parquet')}'")
con.sql("SELECT COUNT(*) AS rows FROM oct_clean").df()

,rows
0,42418544


In [43]:
# Clean November with the same rules as October (runs only when rebuilding data)
if RUN_DATA_PREP:
    nov_src = find_file('2019-Nov.csv')
    con.sql(f"""
        COPY (
            SELECT DISTINCT
                event_time,
                CAST(event_time AS DATE)              AS event_date,
                HOUR(event_time)                      AS event_hour,
                DAYOFWEEK(event_time)                 AS day_of_week,
                event_type, product_id, category_id, category_code,
                split_part(category_code, '.', 1)     AS category_l1,
                COALESCE(brand, 'unknown')            AS brand,
                price, user_id, user_session
            FROM read_csv_auto('{nov_src}')
        ) TO '/kaggle/working/nov_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
    """)

con.sql(f"CREATE OR REPLACE VIEW nov_clean AS SELECT * FROM '{find_file('nov_clean.parquet')}'")
con.sql("SELECT COUNT(*) AS rows, MIN(event_time) AS first_event, MAX(event_time) AS last_event FROM nov_clean").df()

,rows,first_event,last_event
0,67401460,2019-11-01,2019-11-30 23:59:59


In [44]:
# Raw November row count, used to calculate duplicates removed (runs only when rebuilding data)
# Recorded result: 67,501,979 raw rows, 100,519 duplicates removed (0.15%)
if RUN_DATA_PREP:
    con.sql(f"SELECT COUNT(*) AS raw_rows FROM read_csv_auto('{find_file('2019-Nov.csv')}')").df()

## 1. Data loading

In [45]:
!pip install -q duckdb

import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        path = os.path.join(root, f)
        print(path, round(os.path.getsize(path) / 1e9, 2), 'GB')


/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct_clean.parquet 1.46 GB
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/nov_clean.parquet 2.44 GB
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__results__.html 0.0 GB
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/oct.parquet 0.75 GB
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__notebook__.ipynb 0.0 GB
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/__output__.json 0.0 GB
/kaggle/input/notebooks/guilinzhao/e-commerce-purchase-prediction/custom.css 0.0 GB
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct_clean.parquet 1.46 GB
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/nov_clean.parquet 2.44 GB
/kaggle/input/datasets/guilinzhao/rees46-clean-oct-nov/oct.parquet 0.75 GB
/kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv 9.01 GB
/kaggle/input/datasets/mkechinov/e

In [46]:
import duckdb

# Find the October file automatically
src = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == '2019-Oct.csv':
            src = os.path.join(root, f)

print('Found:', src)

con = duckdb.connect()
con.sql("SET memory_limit='20GB'")
con.sql("SET threads=4")
con.sql("SET temp_directory='/tmp/duckdb'")

con.sql(f"""
    COPY (SELECT * FROM read_csv_auto('{src}'))
    TO '/kaggle/working/oct.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")
print('done')

Found: /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Oct.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

done


## 2. Data quality audit

In [47]:
con.sql("CREATE OR REPLACE VIEW oct AS SELECT * FROM '/kaggle/working/oct.parquet'")

# 1. Overall scale
con.sql("""
    SELECT COUNT(*) AS rows,
           MIN(event_time) AS first_event,
           MAX(event_time) AS last_event,
           COUNT(DISTINCT user_id) AS users,
           COUNT(DISTINCT user_session) AS sessions,
           COUNT(DISTINCT product_id) AS products
    FROM oct
""").show()

# 2. Event type distribution
con.sql("""
    SELECT event_type, COUNT(*) AS n,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM oct GROUP BY 1 ORDER BY 2 DESC
""").show()

# 3. Missing values
con.sql("""
    SELECT ROUND(100 * AVG(CASE WHEN category_code IS NULL THEN 1 ELSE 0 END), 2) AS category_null_pct,
           ROUND(100 * AVG(CASE WHEN brand IS NULL THEN 1 ELSE 0 END), 2) AS brand_null_pct,
           ROUND(100 * AVG(CASE WHEN user_session IS NULL THEN 1 ELSE 0 END), 2) AS session_null_pct
    FROM oct
""").show()

# 4. Exact duplicate rows
con.sql("""
    SELECT (SELECT COUNT(*) FROM oct) - (SELECT COUNT(*) FROM (SELECT DISTINCT * FROM oct)) AS duplicate_rows
""").show()

# 5. Unusual prices
con.sql("""
    SELECT COUNT(*) FILTER (WHERE price <= 0) AS zero_or_negative_price,
           MIN(price) AS min_price, MAX(price) AS max_price
    FROM oct
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┬─────────────────────┬─────────────────────┬─────────┬──────────┬──────────┐
│   rows   │     first_event     │     last_event      │  users  │ sessions │ products │
│  int64   │      timestamp      │      timestamp      │  int64  │  int64   │  int64   │
├──────────┼─────────────────────┼─────────────────────┼─────────┼──────────┼──────────┤
│ 42448764 │ 2019-10-01 00:00:00 │ 2019-10-31 23:59:59 │ 3022290 │  9244421 │   166794 │
└──────────┴─────────────────────┴─────────────────────┴─────────┴──────────┴──────────┘

┌────────────┬──────────┬────────┐
│ event_type │    n     │  pct   │
│  varchar   │  int64   │ double │
├────────────┼──────────┼────────┤
│ view       │ 40779399 │  96.07 │
│ cart       │   926516 │   2.18 │
│ purchase   │   742849 │   1.75 │
└────────────┴──────────┴────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────────────┬────────────────┬──────────────────┐
│ category_null_pct │ brand_null_pct │ session_null_pct │
│      double       │     double     │      double      │
├───────────────────┼────────────────┼──────────────────┤
│             31.84 │           14.4 │              0.0 │
└───────────────────┴────────────────┴──────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────┐
│ duplicate_rows │
│     int64      │
├────────────────┤
│          30220 │
└────────────────┘

┌────────────────────────┬───────────┬───────────┐
│ zero_or_negative_price │ min_price │ max_price │
│         int64          │  double   │  double   │
├────────────────────────┼───────────┼───────────┤
│                  68673 │       0.0 │   2574.07 │
└────────────────────────┴───────────┴───────────┘



In [48]:
# A. Zero prices by event type
con.sql("""
    SELECT event_type, COUNT(*) AS zero_price_rows
    FROM oct WHERE price = 0
    GROUP BY 1 ORDER BY 2 DESC
""").show()

# B. Is category_id complete, even where category_code is missing?
con.sql("""
    SELECT ROUND(100 * AVG(CASE WHEN category_id IS NULL THEN 1 ELSE 0 END), 2) AS category_id_null_pct
    FROM oct
""").show()

# C. How many purchases had no cart event in the same session?
con.sql("""
    WITH p AS (SELECT DISTINCT user_session, product_id FROM oct WHERE event_type = 'purchase'),
         c AS (SELECT DISTINCT user_session, product_id FROM oct WHERE event_type = 'cart')
    SELECT COUNT(*) AS purchases,
           COUNT(*) FILTER (WHERE c.user_session IS NULL) AS purchases_without_cart,
           ROUND(100.0 * COUNT(*) FILTER (WHERE c.user_session IS NULL) / COUNT(*), 1) AS pct_without_cart
    FROM p LEFT JOIN c USING (user_session, product_id)
""").show()

┌────────────┬─────────────────┐
│ event_type │ zero_price_rows │
│  varchar   │      int64      │
├────────────┼─────────────────┤
│ view       │           68523 │
│ cart       │             150 │
└────────────┴─────────────────┘

┌──────────────────────┐
│ category_id_null_pct │
│        double        │
├──────────────────────┤
│                  0.0 │
└──────────────────────┘



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────────────────┬──────────────────┐
│ purchases │ purchases_without_cart │ pct_without_cart │
│   int64   │         int64          │      double      │
├───────────┼────────────────────────┼──────────────────┤
│    690372 │                 385432 │             55.8 │
└───────────┴────────────────────────┴──────────────────┘



In [49]:
con.sql("""
    WITH p AS (SELECT DISTINCT user_id, product_id FROM oct WHERE event_type = 'purchase'),
         c AS (SELECT DISTINCT user_id, product_id FROM oct WHERE event_type = 'cart')
    SELECT ROUND(100.0 * COUNT(*) FILTER (WHERE c.user_id IS NULL) / COUNT(*), 1) AS pct_never_carted
    FROM p LEFT JOIN c USING (user_id, product_id)
""").show()

┌──────────────────┐
│ pct_never_carted │
│      double      │
├──────────────────┤
│             53.5 │
└──────────────────┘



## 3. Cleaning decisions
Exact duplicates are removed, zero prices are kept but excluded from price-based features, category_id is used for grouping, and missing brands are labeled "unknown". The same rules are applied to both months.

In [50]:
con.sql("""
    COPY (
        SELECT DISTINCT
            event_time,
            CAST(event_time AS DATE)                 AS event_date,
            HOUR(event_time)                         AS event_hour,
            DAYOFWEEK(event_time)                    AS day_of_week,
            event_type,
            product_id,
            category_id,
            category_code,
            split_part(category_code, '.', 1)        AS category_l1,
            COALESCE(brand, 'unknown')               AS brand,
            price,
            user_id,
            user_session
        FROM oct
    ) TO '/kaggle/working/oct_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.sql("CREATE OR REPLACE VIEW oct_clean AS SELECT * FROM '/kaggle/working/oct_clean.parquet'")
con.sql("SELECT COUNT(*) AS rows FROM oct_clean").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌──────────┐
│   rows   │
│  int64   │
├──────────┤
│ 42418544 │
└──────────┘



October 2019 contains 42.4M events from 3.0M users across 9.2M sessions. Views dominate (96%). 55.8% of purchases had no cart event in the same session, so carting is not a required step and a standard view → cart → purchase funnel would be misleading. category_code is missing for 31.8% of events, but category_id is complete and is used for grouping. brand is missing for 14.4% and is labeled “unknown.” 30,220 exact duplicate rows (0.07%) were removed. 68,673 zero-price events are all views or carts, so revenue is unaffected; they are excluded only from price-based features.

55.8% of purchases had no cart event in the same session, and 53.5% of purchased products were never carted by that user during October. Cross-session carting is therefore rare, and carting is not a required step before purchase, so a standard view → cart → purchase funnel would be misleading.

Stage 2: define the prediction problem

## 4. Prediction setup
Training: features from Oct 1–24, label = purchase on Oct 25–31.
Test: features from Oct 8–31, label = purchase on Nov 1–7.
Both feature windows are 24 days, and features never overlap the label window.

In [51]:
# Step 1: build the training labels
con.sql("""
    CREATE OR REPLACE TABLE train_base AS
    WITH feature_users AS (
        SELECT DISTINCT user_id
        FROM oct_clean
        WHERE event_time < TIMESTAMP '2019-10-25'
    ),
    buyers AS (
        SELECT DISTINCT user_id
        FROM oct_clean
        WHERE event_type = 'purchase'
          AND event_time >= TIMESTAMP '2019-10-25'
    )
    SELECT f.user_id,
           CASE WHEN b.user_id IS NOT NULL THEN 1 ELSE 0 END AS purchased_next_7d
    FROM feature_users f
    LEFT JOIN buyers b USING (user_id)
""")

con.sql("""
    SELECT COUNT(*) AS users,
           SUM(purchased_next_7d) AS buyers,
           ROUND(100 * AVG(purchased_next_7d), 2) AS buyer_pct
    FROM train_base
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌─────────┬────────┬───────────┐
│  users  │ buyers │ buyer_pct │
│  int64  │ int128 │  double   │
├─────────┼────────┼───────────┤
│ 2559818 │  62181 │      2.43 │
└─────────┴────────┴───────────┘



## 5. Feature engineering

In [52]:
# Step 2: build the features

def build_features(start, cutoff, source='oct_clean'):
    # Only events in [start, cutoff) are used, so no feature can see the label window
    return f"""
    WITH base AS (
        SELECT user_id,
            COUNT(*)                                                  AS n_events,
            COUNT(*) FILTER (WHERE event_type = 'view')               AS n_views,
            COUNT(*) FILTER (WHERE event_type = 'cart')               AS n_carts,
            COUNT(*) FILTER (WHERE event_type = 'purchase')           AS n_purchases,
            COUNT(DISTINCT user_session)                              AS n_sessions,
            COUNT(DISTINCT event_date)                                AS active_days,
            COUNT(DISTINCT product_id)                                AS n_products,
            COUNT(DISTINCT category_id)                               AS n_categories,
            COUNT(DISTINCT brand)                                     AS n_brands,

            -- Recency, in days before the cutoff
            DATE_DIFF('hour', MAX(event_time), TIMESTAMP '{cutoff}') / 24.0                                       AS days_since_last_event,
            DATE_DIFF('hour', MAX(event_time) FILTER (WHERE event_type = 'cart'), TIMESTAMP '{cutoff}') / 24.0     AS days_since_last_cart,
            DATE_DIFF('hour', MAX(event_time) FILTER (WHERE event_type = 'purchase'), TIMESTAMP '{cutoff}') / 24.0 AS days_since_last_purchase,

            -- Recent activity: the 7 days right before the cutoff
            COUNT(*) FILTER (WHERE event_time >= TIMESTAMP '{cutoff}' - INTERVAL 7 DAY)                            AS n_events_last7d,
            COUNT(*) FILTER (WHERE event_type = 'cart' AND event_time >= TIMESTAMP '{cutoff}' - INTERVAL 7 DAY)    AS n_carts_last7d,

            -- Spending; zero prices excluded from price averages (all zero prices are views or carts)
            COALESCE(SUM(price) FILTER (WHERE event_type = 'purchase'), 0)  AS revenue,
            AVG(price) FILTER (WHERE event_type = 'view' AND price > 0)     AS avg_view_price
        FROM {source}
        WHERE event_time >= TIMESTAMP '{start}' AND event_time < TIMESTAMP '{cutoff}'
        GROUP BY user_id
    )
    SELECT *,
        n_carts::DOUBLE     / NULLIF(n_views, 0)    AS cart_per_view,
        n_purchases::DOUBLE / NULLIF(n_views, 0)    AS purchase_per_view,
        n_events::DOUBLE    / NULLIF(n_sessions, 0) AS events_per_session
    FROM base
    """

con.sql(f"""
    CREATE OR REPLACE TABLE train_features AS
    SELECT b.user_id, b.purchased_next_7d, f.* EXCLUDE (user_id)
    FROM train_base b
    JOIN ({build_features('2019-10-01', '2019-10-25')}) f USING (user_id)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [53]:
# Step 3: two checks
# Reconciliation: should equal 2,559,818, same as train_base
con.sql("SELECT COUNT(*) AS rows FROM train_features").show()

# Sanity check: do buyers and non-buyers look different?
con.sql("""
    SELECT purchased_next_7d,
           COUNT(*)                                                     AS users,
           ROUND(AVG(n_events), 1)                                      AS avg_events,
           ROUND(AVG(n_sessions), 1)                                    AS avg_sessions,
           ROUND(AVG(days_since_last_event), 1)                         AS avg_days_since_last,
           ROUND(100 * AVG(CASE WHEN n_carts > 0 THEN 1 ELSE 0 END), 1)     AS pct_ever_carted,
           ROUND(100 * AVG(CASE WHEN n_purchases > 0 THEN 1 ELSE 0 END), 1) AS pct_prior_buyers
    FROM train_features
    GROUP BY 1 ORDER BY 1
""").show()

┌─────────┐
│  rows   │
│  int64  │
├─────────┤
│ 2559818 │
└─────────┘

┌───────────────────┬─────────┬────────────┬──────────────┬─────────────────────┬─────────────────┬──────────────────┐
│ purchased_next_7d │  users  │ avg_events │ avg_sessions │ avg_days_since_last │ pct_ever_carted │ pct_prior_buyers │
│       int32       │  int64  │   double   │    double    │       double        │     double      │      double      │
├───────────────────┼─────────┼────────────┼──────────────┼─────────────────────┼─────────────────┼──────────────────┤
│                 0 │ 2497637 │       12.3 │          2.7 │                 9.7 │            10.2 │             10.2 │
│                 1 │   62181 │       39.8 │          6.9 │                 5.5 │            41.9 │             49.3 │
└───────────────────┴─────────┴────────────┴──────────────┴─────────────────────┴─────────────────┴──────────────────┘



In [54]:
# Step 4: prepare November and build the test set

import os

# Find the November file automatically
nov_src = None
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f == '2019-Nov.csv':
            nov_src = os.path.join(root, f)
print('Found:', nov_src)

# Same cleaning as October: remove exact duplicates, add derived columns
con.sql(f"""
    COPY (
        SELECT DISTINCT
            event_time,
            CAST(event_time AS DATE)              AS event_date,
            HOUR(event_time)                      AS event_hour,
            DAYOFWEEK(event_time)                 AS day_of_week,
            event_type, product_id, category_id, category_code,
            split_part(category_code, '.', 1)     AS category_l1,
            COALESCE(brand, 'unknown')            AS brand,
            price, user_id, user_session
        FROM read_csv_auto('{nov_src}')
    ) TO '/kaggle/working/nov_clean.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

con.sql("CREATE OR REPLACE VIEW nov_clean AS SELECT * FROM '/kaggle/working/nov_clean.parquet'")
con.sql("SELECT COUNT(*) AS rows, MIN(event_time) AS first_event, MAX(event_time) AS last_event FROM nov_clean").df()

Found: /kaggle/input/datasets/mkechinov/ecommerce-behavior-data-from-multi-category-store/2019-Nov.csv


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,first_event,last_event
0,67401460,2019-11-01,2019-11-30 23:59:59


### Test set

In [55]:
# Test set: features from Oct 8-31, label = purchase on Nov 1-7
con.sql("""
    CREATE OR REPLACE TABLE test_base AS
    WITH feature_users AS (
        SELECT DISTINCT user_id FROM oct_clean
        WHERE event_time >= TIMESTAMP '2019-10-08'
    ),
    buyers AS (
        SELECT DISTINCT user_id FROM nov_clean
        WHERE event_type = 'purchase'
          AND event_time < TIMESTAMP '2019-11-08'
    )
    SELECT f.user_id,
           CASE WHEN b.user_id IS NOT NULL THEN 1 ELSE 0 END AS purchased_next_7d
    FROM feature_users f
    LEFT JOIN buyers b USING (user_id)
""")

# Same feature function as training, shifted forward by one week
con.sql(f"""
    CREATE OR REPLACE TABLE test_features AS
    SELECT b.user_id, b.purchased_next_7d, f.* EXCLUDE (user_id)
    FROM test_base b
    JOIN ({build_features('2019-10-08', '2019-11-01')}) f USING (user_id)
""")

con.sql("""
    SELECT COUNT(*) AS users,
           SUM(purchased_next_7d) AS buyers,
           ROUND(100 * AVG(purchased_next_7d), 2) AS buyer_pct
    FROM test_features
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,users,buyers,buyer_pct
0,2596037,66974.0,2.58


Buyers are more active, more recent, and far more likely to have carted or bought before. Prior purchase is the strongest single signal, but about half of next-week buyers had never purchased, so the model must also identify first-time buyers from browsing behavior.

In [56]:
con.sql(f"SELECT COUNT(*) AS raw_rows FROM read_csv_auto('{nov_src}')").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,raw_rows
0,67501979


## 6. Baseline models
Before training a complex model, simple baselines show what "good" looks like. Any model must clearly beat these to be worth using.

In [57]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

# Load the feature tables into pandas for modeling
train = con.sql("SELECT * FROM train_features").df()
test = con.sql("SELECT * FROM test_features").df()
y_train = train['purchased_next_7d']
y_test = test['purchased_next_7d']

print('Train:', train.shape, ' Test:', test.shape)

Train: (2559818, 21)  Test: (2596037, 21)


### Evaluation metrics
Accuracy is not used because only about 2.5% of users buy. Instead:
- ROC-AUC: how well the model ranks buyers above non-buyers overall (0.5 = random)
- PR-AUC: ranking quality focused on the rare buyers (random = the buyer rate, about 0.026)
- Lift at top 10%: how many times more buyers are found in the top-scored 10% of users than in a random 10%

In [60]:
def lift_at(y_true, scores, top_frac=0.10):
    # Buyer rate among the top-scored users, divided by the overall buyer rate
    y = np.asarray(y_true)
    n_top = int(len(y) * top_frac)
    order = np.argsort(-np.asarray(scores))
    return y[order[:n_top]].mean() / y.mean()

def evaluate(name, y_true, scores):
    return {
        'model': name,
        'roc_auc': round(roc_auc_score(y_true, scores), 4),
        'pr_auc': round(average_precision_score(y_true, scores), 4),
        'lift_top10': round(lift_at(y_true, scores, 0.10), 2),
    }

### Rule-based baselines
Two simple rules a business might use without any model: contact the most recently active users first, or contact past buyers first.

In [61]:
results = []

# Random guessing: PR-AUC equals the buyer rate, lift equals 1
results.append({'model': 'Random', 'roc_auc': 0.5,
                'pr_auc': round(y_test.mean(), 4), 'lift_top10': 1.0})

# Rule 1: most recently active users first
results.append(evaluate('Rule: recency', y_test, -test['days_since_last_event']))

# Rule 2: past buyers first, ties broken by recency
# (days_since_last_event is at most 24, so multiplying by 100 makes past buyers always rank first)
rule2 = (test['n_purchases'] > 0).astype(int) * 100 - test['days_since_last_event']
results.append(evaluate('Rule: past buyers + recency', y_test, rule2))

pd.DataFrame(results)

,model,roc_auc,pr_auc,lift_top10
0,Random,0.5000,0.0258,1.00
1,Rule: recency,0.6677,0.0511,2.49
2,Rule: past buyers + recency,0.7659,0.1231,4.77


### Logistic regression baseline
A simple statistical model using all features. Counts are log-transformed because they are highly skewed, and missing values (for example, "days since last cart" for users who never carted) are filled with the median plus a flag marking that the value was missing.

In [62]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler

features = [c for c in train.columns if c not in ('user_id', 'purchased_next_7d')]

logreg = make_pipeline(
    SimpleImputer(strategy='median', add_indicator=True),  # fill missing values and flag them
    FunctionTransformer(np.log1p),                         # reduce skew in counts
    StandardScaler(),
    LogisticRegression(max_iter=1000),
)
logreg.fit(train[features], y_train)

# Score the November test set
results.append(evaluate('Logistic regression', y_test, logreg.predict_proba(test[features])[:, 1]))
pd.DataFrame(results)

,model,roc_auc,pr_auc,lift_top10
0,Random,0.5000,0.0258,1.00
1,Rule: recency,0.6677,0.0511,2.49
2,Rule: past buyers + recency,0.7659,0.1231,4.77
3,Logistic regression,0.7891,0.1918,4.78


In [63]:
# Compare lift at different cutoffs to see where the model's advantage comes from
rule2_scores = rule2
logreg_scores = logreg.predict_proba(test[features])[:, 1]

pd.DataFrame({
    'top_pct': ['1%', '5%', '10%', '20%'],
    'rule2_lift':  [round(lift_at(y_test, rule2_scores, p), 2)  for p in (0.01, 0.05, 0.10, 0.20)],
    'logreg_lift': [round(lift_at(y_test, logreg_scores, p), 2) for p in (0.01, 0.05, 0.10, 0.20)],
})

,top_pct,rule2_lift,logreg_lift
0,1%,9.11,14.76
1,5%,6.43,6.94
2,10%,4.77,4.78
3,20%,3.06,3.18


### Baseline results
Prioritizing past buyers is a strong rule: its top 10% contains 4.77 times more buyers than a random 10%. Logistic regression improves PR-AUC from 0.123 to 0.192 (+56%). Its advantage is concentrated at the very top of the ranking: in the top 1% of users, lift rises from 9.11 to 14.76, meaning about 38% of those users buy within a week, versus about 24% under the rule. At 10% the two are equal, because both select mostly past buyers, and at 20% the model is only slightly better, which suggests it still struggles to identify first-time buyers.

## 7. Gradient boosting model
LightGBM can learn non-linear patterns and feature combinations that logistic regression cannot, which may help identify first-time buyers. It handles missing values natively, so no imputation or scaling is needed.

In [64]:
import lightgbm as lgb
from sklearn.model_selection import train_test_split

# Hold out 20% of the training users to decide when to stop adding trees.
# The November test set is never used for any training decision.
X_tr, X_val, y_tr, y_val = train_test_split(
    train[features], y_train, test_size=0.2, random_state=42, stratify=y_train
)

gbm = lgb.LGBMClassifier(
    n_estimators=2000,       # upper limit; early stopping picks the actual number
    learning_rate=0.05,
    num_leaves=63,
    min_child_samples=200,   # avoid tiny leaves that memorize noise
    subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8,
    random_state=42,
    verbose=-1,
)
gbm.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

gbm_scores = gbm.predict_proba(test[features])[:, 1]
results.append(evaluate('LightGBM', y_test, gbm_scores))
pd.DataFrame(results)

Training until validation scores don't improve for 100 rounds
[100]	valid_0's average_precision: 0.214113	valid_0's binary_logloss: 0.093016
[200]	valid_0's average_precision: 0.214513	valid_0's binary_logloss: 0.0929896
Early stopping, best iteration is:
[166]	valid_0's average_precision: 0.214445	valid_0's binary_logloss: 0.0929861


,model,roc_auc,pr_auc,lift_top10
0,Random,0.5000,0.0258,1.00
1,Rule: recency,0.6677,0.0511,2.49
2,Rule: past buyers + recency,0.7659,0.1231,4.77
3,Logistic regression,0.7891,0.1918,4.78
4,LightGBM,0.8003,0.2047,4.96


In [65]:
# Lift at different cutoffs for all three approaches
pd.DataFrame({
    'top_pct':     ['1%', '5%', '10%', '20%'],
    'rule2_lift':  [round(lift_at(y_test, rule2_scores, p), 2)  for p in (0.01, 0.05, 0.10, 0.20)],
    'logreg_lift': [round(lift_at(y_test, logreg_scores, p), 2) for p in (0.01, 0.05, 0.10, 0.20)],
    'gbm_lift':    [round(lift_at(y_test, gbm_scores, p), 2)    for p in (0.01, 0.05, 0.10, 0.20)],
})

,top_pct,rule2_lift,logreg_lift,gbm_lift
0,1%,9.11,14.76,15.46
1,5%,6.43,6.94,7.21
2,10%,4.77,4.78,4.96
3,20%,3.06,3.18,3.25


### Model comparison
LightGBM is the best model at every cutoff (PR-AUC 0.205, ROC-AUC 0.800), and is the first to clearly beat the past-buyer rule at the top 10% (lift 4.96 vs. 4.77). Validation PR-AUC (0.214) and November test PR-AUC (0.205) are close, so the model generalizes to a later period without obvious overfitting. However, the gain over logistic regression is modest (about +7% in PR-AUC), suggesting the current features, rather than the model, limit performance. The next check focuses on users who had never purchased, where the rule has no signal.

In [66]:
# Focus on users who had never purchased before the cutoff
never_bought = (test['n_purchases'] == 0).values
seg_y = y_test.values[never_bought]

print(f"Users who never purchased: {never_bought.mean():.1%} of test users")
print(f"Their buyer rate next week: {seg_y.mean():.2%}")
print(f"Share of all next-week buyers who are first-time buyers: {seg_y.sum() / y_test.sum():.1%}")

pd.DataFrame([
    evaluate('Logistic regression (never purchased)', seg_y, logreg_scores[never_bought]),
    evaluate('LightGBM (never purchased)', seg_y, gbm_scores[never_bought]),
])

Users who never purchased: 88.9% of test users
Their buyer rate next week: 1.46%
Share of all next-week buyers who are first-time buyers: 50.4%


,model,roc_auc,pr_auc,lift_top10
0,Logistic regression (never purchased),0.6810,0.0357,2.90
1,LightGBM (never purchased),0.6993,0.0406,3.03


### First-time buyers
88.9% of test users had never purchased, and their next-week buyer rate is only 1.46%, yet they account for 50.4% of all next-week buyers. Both models find signal in this group, and LightGBM's advantage is larger here (+14% PR-AUC over logistic regression) than across all users (+7%). But absolute performance remains weak (PR-AUC 0.041, lift 3.03 at the top 10%), so the current activity-based features carry limited information about first-time purchase intent. Next step: engineer intent-focused features.

## 8. Intent-focused features
The current features describe how active a user is. These new features aim to capture purchase intent, which matters most for users who have never bought: repeated views of the same product, items carted but not purchased, focus on one category, and behavior in the most recent days and session.

In [68]:
def build_intent_features(start, cutoff, source='oct_clean'):
    # Same window rule as build_features: only events in [start, cutoff)
    return f"""
    WITH w AS (
        SELECT * FROM {source}
        WHERE event_time >= TIMESTAMP '{start}' AND event_time < TIMESTAMP '{cutoff}'
    ),
    -- One row per user and product
    prod AS (
        SELECT user_id, product_id,
               COUNT(*) FILTER (WHERE event_type = 'view')                AS views,
               MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END)       AS carted,
               MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END)   AS bought
        FROM w GROUP BY 1, 2
    ),
    prod_agg AS (
        SELECT user_id,
               MAX(views)                                          AS max_views_one_product,
               COUNT(*) FILTER (WHERE views >= 3)                  AS n_products_viewed_3plus,
               COUNT(*) FILTER (WHERE carted = 1 AND bought = 0)   AS open_cart_products
        FROM prod GROUP BY 1
    ),
    -- Share of views in the user's most-viewed category
    cat AS (
        SELECT user_id, category_id, COUNT(*) AS views
        FROM w WHERE event_type = 'view' GROUP BY 1, 2
    ),
    cat_agg AS (
        SELECT user_id, MAX(views)::DOUBLE / SUM(views) AS top_category_share
        FROM cat GROUP BY 1
    ),
    -- Very recent behavior: last 3 days before the cutoff
    recent AS (
        SELECT user_id,
               COUNT(*) FILTER (WHERE event_time >= TIMESTAMP '{cutoff}' - INTERVAL 3 DAY)                         AS n_events_last3d,
               COUNT(*) FILTER (WHERE event_type = 'cart' AND event_time >= TIMESTAMP '{cutoff}' - INTERVAL 3 DAY) AS n_carts_last3d
        FROM w GROUP BY 1
    ),
    -- The user's most recent session
    sess AS (
        SELECT user_id, user_session,
               MAX(event_time)                                       AS last_time,
               COUNT(*)                                              AS n,
               MAX(CASE WHEN event_type = 'cart' THEN 1 ELSE 0 END)  AS has_cart
        FROM w GROUP BY 1, 2
    ),
    last_sess AS (
        SELECT user_id, n AS last_session_events, has_cart AS last_session_cart
        FROM sess
        QUALIFY ROW_NUMBER() OVER (PARTITION BY user_id ORDER BY last_time DESC) = 1
    )
    SELECT r.user_id,
           p.max_views_one_product, p.n_products_viewed_3plus, p.open_cart_products,
           c.top_category_share,
           r.n_events_last3d, r.n_carts_last3d,
           l.last_session_events, l.last_session_cart
    FROM recent r
    LEFT JOIN prod_agg  p USING (user_id)
    LEFT JOIN cat_agg   c USING (user_id)
    LEFT JOIN last_sess l USING (user_id)
    """

# Build for the same windows as before: training Oct 1-24, test Oct 8-31
con.sql(f"CREATE OR REPLACE TABLE train_intent AS {build_intent_features('2019-10-01', '2019-10-25')}")
con.sql(f"CREATE OR REPLACE TABLE test_intent AS {build_intent_features('2019-10-08', '2019-11-01')}")
print('intent features built')

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

intent features built


In [69]:
# Add the new features to the existing tables (same row order as before)
new_features = ['max_views_one_product', 'n_products_viewed_3plus', 'open_cart_products',
                'top_category_share', 'n_events_last3d', 'n_carts_last3d',
                'last_session_events', 'last_session_cart', 'recent_share_7d']

train2 = train.merge(con.sql("SELECT * FROM train_intent").df(), on='user_id', how='left')
test2 = test.merge(con.sql("SELECT * FROM test_intent").df(), on='user_id', how='left')

# Share of all activity that happened in the last 7 days: is engagement accelerating?
for df in (train2, test2):
    df['recent_share_7d'] = df['n_events_last7d'] / df['n_events']

features_v2 = features + new_features

# Reconciliation: row counts must not change after the merge
print('Train:', len(train), '->', len(train2), '  Test:', len(test), '->', len(test2))

Train: 2559818 -> 2559818   Test: 2596037 -> 2596037


In [70]:
# Signal check among users who had never purchased: do next-week buyers look different?
seg = train2[train2['n_purchases'] == 0]
seg.groupby('purchased_next_7d')[new_features].mean().round(3).T

purchased_next_7d,0,1
max_views_one_product,2.422,3.940
n_products_viewed_3plus,0.713,1.916
open_cart_products,0.057,0.201
top_category_share,0.834,0.727
n_events_last3d,1.231,5.128
n_carts_last3d,0.013,0.096
last_session_events,3.536,4.972
last_session_cart,0.024,0.066
recent_share_7d,0.320,0.427


### Intent feature signal check
Among users who had never purchased, next-week buyers differ clearly on the new features. The strongest signals are very recent: carts in the last 3 days (0.096 vs. 0.013, about 7x) and events in the last 3 days (5.1 vs. 1.2, about 4x). Open cart items and repeatedly viewed products also separate the groups. One result runs against intuition: buyers have a lower top-category share (0.73 vs. 0.83), likely because users with very few views mechanically have a share near 100%, so this feature mostly reflects low activity rather than focus. Means overlap substantially, so the real test is whether the model improves.

In [71]:
# Retrain LightGBM with the same settings and the same validation split, adding only the new features.
# Keeping everything else identical means any change in performance comes from the features.
X_tr2, X_val2, y_tr2, y_val2 = train_test_split(
    train2[features_v2], y_train, test_size=0.2, random_state=42, stratify=y_train
)

gbm2 = lgb.LGBMClassifier(**gbm.get_params())
gbm2.fit(
    X_tr2, y_tr2,
    eval_set=[(X_val2, y_val2)],
    eval_metric='average_precision',
    callbacks=[lgb.early_stopping(100), lgb.log_evaluation(100)],
)

gbm2_scores = gbm2.predict_proba(test2[features_v2])[:, 1]

Training until validation scores don't improve for 100 rounds
[100]	valid_0's average_precision: 0.215961	valid_0's binary_logloss: 0.0928872
[200]	valid_0's average_precision: 0.216234	valid_0's binary_logloss: 0.0928419
Early stopping, best iteration is:
[166]	valid_0's average_precision: 0.216439	valid_0's binary_logloss: 0.0928422


In [72]:
# Compare v1 (original features) and v2 (with intent features), for all users and for never-purchased users
comparison = pd.DataFrame([
    evaluate('LightGBM v1, all users',        y_test, gbm_scores),
    evaluate('LightGBM v2, all users',        y_test, gbm2_scores),
    evaluate('LightGBM v1, never purchased',  seg_y,  gbm_scores[never_bought]),
    evaluate('LightGBM v2, never purchased',  seg_y,  gbm2_scores[never_bought]),
])
comparison

,model,roc_auc,pr_auc,lift_top10
0,"LightGBM v1, all users",0.8003,0.2047,4.96
1,"LightGBM v2, all users",0.8008,0.2065,4.99
2,"LightGBM v1, never purchased",0.6993,0.0406,3.03
3,"LightGBM v2, never purchased",0.6998,0.0410,3.04


### Intent features: result
Adding the intent features changed PR-AUC by only about 1% (0.2047 to 0.2065 for all users, 0.0406 to 0.0410 for never-purchased users), which is within normal run-to-run variation. Although the new features separate buyers from non-buyers on their own, their information largely overlaps with existing features such as 7-day carts, recency, and cart counts, which LightGBM already combines. This suggests that predicting first-time purchases from browsing behavior alone is close to its limit; the missing signal likely lives in data not available here, such as marketing exposure, price changes, or user attributes. Since v2 adds 9 features for no meaningful gain, v1 is kept as the final model.